# Latent → Feature Mapping: Load Saved Scores and Reconstruct the Visualisation

`07_attribution_vis_all.py`'s `plot_latent_feature_mapping` (reused as-is by `07b_cross_dataset_attribution_vis.py`)
correlates each model's latent dimensions against kinetic curve features (Spearman, MI, cosine similarity),
combines them into one score, and draws a figure showing only the top-N latents whose best-matching feature
is unique. When called with `return_scores=True`, it ALSO returns every `(branch, latent_rank, feature)`
triple's raw scores plus the saliency profiles needed to redraw the figure — not just the winning feature per
latent that ends up in the PNG. The pipeline (`run_interpretation_pipeline` in 07, `run_fold` in 07b) saves
this as a joblib bundle alongside the existing PNGs.

This notebook shows how to:
1. Load that saved joblib bundle (`{"scores": DataFrame, "profiles": DataFrame, "curve_meta": dict}`).
2. Explore the full correlation table directly — e.g. "what was the 2nd/3rd best feature for this latent"
   without rerunning anything.
3. Reconstruct the exact same PNG by rebuilding the `sections` structure `plot_latent_feature_mapping` uses
   internally, then calling the same `render_latent_feature_mapping_figure` the live pipeline calls —
   no model inference, no GPU, no rerun.

**Known limitation:** the saved bundle does not persist the raw kinetic feature matrix (`feat_matrix`), so
the reconstructed figure omits the one minor visual detail that depends on it — the dotted vertical
"best-match timestamp" marker line drawn only for timing-type features. Everything else (curves, saliency
overlay, score bars, star annotation, legend, colorbar) reconstructs exactly.

In [ ]:
import os

try:
    # VS Code injects '__vsc_ipynb_file__' into the globals automatically
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(notebook_path)
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
import importlib.util
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, 'utils/model_training')
import config

# Import 07's module by file path (it starts with a digit, so it isn't a valid
# `import` target) -- same approach 07b itself uses to reuse 07's functions.
_spec = importlib.util.spec_from_file_location('attrib07', '07_attribution_vis_all.py')
attrib07 = importlib.util.module_from_spec(_spec)
sys.argv = ['notebook']  # 07's module-level code reads sys.argv in some helpers; keep it harmless
_spec.loader.exec_module(attrib07)

render_latent_feature_mapping_figure = attrib07.render_latent_feature_mapping_figure

## 1. Locate and load a saved scores+profiles+curve_meta bundle

- **07** (single experiment): `{exp_folder}/model_interpretation/{exp_name}/latent_feature_scores.joblib`
  (one file per experiment, accumulating across every `--filter_key`/`--curve_type` you've run with `07`).
- **07b** (cross-dataset LOFO): `{exp_folder}/cross_dataset_cv/{group_name}/model_interpretation/{fold_label}/latent_feature_scores_{curve_type}.joblib`
  (one file per fold **and** curve_type — curve_type is in the filename, not just a column, because
  concurrent SLURM array tasks can process the same fold with different curve_types).

Set `SCORES_PATH` below to whichever one you want to inspect.

In [ ]:
# Example: a single-experiment (07) bundle. Adjust to point at an actual run's output,
# or swap in a 07b-style path (see markdown above) -- the loading/reconstruction code
# below is identical either way, since both share the same {"scores","profiles","curve_meta"} schema.
EXP_FOLDER = config.DEFAULT_EXP_FOLDER
EXP_NAME = "CHANGE_ME"  # e.g. one of os.listdir(EXP_FOLDER)

SCORES_PATH = config.get_viz_dir(Path(EXP_FOLDER), "model_interpretation") / EXP_NAME / "latent_feature_scores.joblib"
print(SCORES_PATH)

bundle = joblib.load(SCORES_PATH)
scores_df = bundle["scores"]
profiles_df = bundle["profiles"]
curve_meta = bundle["curve_meta"]   # dict keyed by curve_type

print(f"scores: {len(scores_df)} rows, profiles: {len(profiles_df)} rows")
print("curve_types available:", list(curve_meta.keys()))
print("models available:", sorted(scores_df['model_name'].unique()))
print("filters available:", sorted(scores_df['filter_key'].unique()))
scores_df.head()

## 2. Explore the raw correlation table directly

Every `(model, filter, curve_type, branch, latent_rank, feature)` combination is here — not just the
winning feature per latent that made it into the PNG. `feature_rank_for_latent` is precomputed (1=best),
so "what was the 2nd/3rd best feature for this latent" is a one-line filter, no re-deriving ranks yourself.

`combined` should always reconstruct from `spearman`/`mi`/`cosine` + the stored weights — useful as a
sanity check that you're reading the table correctly:
`combined == w_spearman*spearman + w_mi*mi + w_cosine*cosine` (NaNs treated as 0, matching the original
computation's skip-on-invalid behaviour).

In [ ]:
MODEL = scores_df['model_name'].iloc[0]
FILTER = scores_df['filter_key'].iloc[0]
CURVE_TYPE = scores_df['curve_type'].iloc[0]

one_run = scores_df[
    (scores_df.model_name == MODEL) & (scores_df.filter_key == FILTER) & (scores_df.curve_type == CURVE_TYPE)
]

# Top 5 features (not just the winner) for the single most important latent of each branch.
for branch in sorted(one_run['branch'].unique()):
    top_latent = one_run[(one_run.branch == branch) & (one_run.latent_rank == 1)]
    print(f"\n--- branch={branch}, latent_rank=1 (the single most important latent) ---")
    display(
        top_latent.sort_values('feature_rank_for_latent')
        [['feature', 'feature_rank_for_latent', 'combined', 'spearman', 'mi', 'mi_raw', 'cosine', 'is_best_feature']]
        .head(5)
    )

In [ ]:
# Sanity check: combined reconstructs from the raw metrics + stored weights.
recon = (
    one_run['w_spearman'] * one_run['spearman'].fillna(0)
    + one_run['w_mi'] * one_run['mi'].fillna(0)
    + one_run['w_cosine'] * one_run['cosine']
)
max_err = (recon - one_run['combined']).abs().max()
print(f"max reconstruction error: {max_err:.2e}")

## 3. Reconstruct the exact figure

`render_latent_feature_mapping_figure` is the same function the live pipeline calls — it just needs the
`sections` structure `plot_latent_feature_mapping` builds internally (one dict per branch: label, the kept
latents' original ranks/flat indices, their saliency profiles, and the full `(k, n_feats)` combined-score
matrix). `rebuild_sections` below reconstructs that structure from `scores_df`/`profiles_df` — filtering to
`kept_in_plot == True` rows (the figure never shows anything else), pivoting scores into the matrix shape,
and pulling saliency profiles from the profiles table.

In [ ]:
def rebuild_sections(scores_df, profiles_df, feat_names, model_name, filter_key, curve_type,
                      group_name=None, fold_label=None):
    """Rebuilds plot_latent_feature_mapping's `sections` list from the saved tables,
    for one (model, filter, curve_type[, group, fold]) combination.

    feat_names must be the SAME ordered list used by the original run (curve_meta's
    "feat_names" entry for this curve_type) -- it controls the score-bar x-axis order
    and feature-group colour-coding, and only contains the "valid" (>=5 finite values)
    features the original run actually scored.
    """
    mask_s = (
        (scores_df.model_name == model_name) & (scores_df.filter_key == filter_key)
        & (scores_df.curve_type == curve_type) & (scores_df.kept_in_plot)
    )
    mask_p = (
        (profiles_df.model_name == model_name) & (profiles_df.filter_key == filter_key)
        & (profiles_df.curve_type == curve_type)
    )
    if group_name is not None:
        mask_s &= (scores_df.group_name == group_name)
        mask_p &= (profiles_df.group_name == group_name)
    if fold_label is not None:
        mask_s &= (scores_df.fold_label == fold_label)
        mask_p &= (profiles_df.fold_label == fold_label)

    sub_s, sub_p = scores_df[mask_s], profiles_df[mask_p]
    feat_to_idx = {f: i for i, f in enumerate(feat_names)}

    sections = []
    for branch in sorted(sub_s['branch'].unique()):
        bs = sub_s[sub_s.branch == branch]
        bp = sub_p[sub_p.branch == branch].set_index('latent_rank')
        true_ranks = sorted(bs['latent_rank'].unique())
        k = len(true_ranks)

        order = np.array([bp.loc[r, 'latent_index'] for r in true_ranks])
        sal_profiles = np.array([bp.loc[r, 'sal_profile'] for r in true_ranks])

        combined = np.zeros((k, len(feat_names)))
        best_feat_idx = np.zeros(k, dtype=int)
        for i, r in enumerate(true_ranks):
            for _, row in bs[bs.latent_rank == r].iterrows():
                j = feat_to_idx[row['feature']]
                combined[i, j] = row['combined']
                if row['is_best_feature']:
                    best_feat_idx[i] = j
        best_score = combined[np.arange(k), best_feat_idx]

        sections.append({
            "label": None if branch == "single" else branch,
            "order": order, "k": k, "true_ranks": true_ranks,
            "sal_profiles": sal_profiles, "combined": combined,
            "best_feat_idx": best_feat_idx, "best_score": best_score,
        })
    return sections

In [ ]:
cm = curve_meta[CURVE_TYPE]
sections = rebuild_sections(scores_df, profiles_df, cm["feat_names"], MODEL, FILTER, CURVE_TYPE)

save_path = f"/tmp/reconstructed_{MODEL}_{FILTER}_{CURVE_TYPE}.png"
render_latent_feature_mapping_figure(
    sections, cm["mean_curve"], cm["std_curve"], cm["timestamps"], cm["feat_names"],
    MODEL, f"Reconstructed ({CURVE_TYPE}/{FILTER})", save_path,
    # feat_matrix intentionally omitted -- not persisted, so the dotted timing-marker
    # line is the one thing that won't reappear; everything else matches exactly.
)
display(Image(filename=save_path))

## 4. Reconstruct for a 07b (cross-dataset LOFO) bundle

Same function, same schema — just pass `group_name`/`fold_label` too, since 07b's tables carry those
extra identifying columns that 07's don't need (one 07 file already = one experiment).

In [ ]:
# Example -- adjust to a real group/fold/curve_type combination you've actually run 07b on.
GROUP_NAME = "CHANGE_ME"
FOLD_LABEL = "CHANGE_ME"
B_CURVE_TYPE = "ori_curve"

b_scores_path = (
    Path(config.MULTI_EXP_FOLDER) / "cross_dataset_cv" / GROUP_NAME / "model_interpretation"
    / FOLD_LABEL / f"latent_feature_scores_{B_CURVE_TYPE}.joblib"
)
print(b_scores_path)

# b_bundle = joblib.load(b_scores_path)
# b_scores_df, b_profiles_df, b_curve_meta = b_bundle["scores"], b_bundle["profiles"], b_bundle["curve_meta"]
# b_model = b_scores_df['model_name'].iloc[0]
# b_filter = b_scores_df['filter_key'].iloc[0]
# b_cm = b_curve_meta[B_CURVE_TYPE]
# b_sections = rebuild_sections(
#     b_scores_df, b_profiles_df, b_cm["feat_names"], b_model, b_filter, B_CURVE_TYPE,
#     group_name=GROUP_NAME, fold_label=FOLD_LABEL,
# )
# render_latent_feature_mapping_figure(
#     b_sections, b_cm["mean_curve"], b_cm["std_curve"], b_cm["timestamps"], b_cm["feat_names"],
#     b_model, f"{GROUP_NAME} | {FOLD_LABEL}", "/tmp/reconstructed_07b.png",
# )
# display(Image(filename="/tmp/reconstructed_07b.png"))